# アンブレラサンプリングの実行

アンブレラサンプリングでは、系に対して目的の場所に留まるように人工的な「傘（Umbrella）＝ポテンシャル」による束縛をかけることで、エネルギーが高い不安定な状態（遷移状態など）など通常の分子シミュレーションではレアイベントとなる構造ももサンプリングすることができる手法となります。

ここまでの工程では反応座標（末端間距離）に沿った初期構造を用意しました。
このノートブックでは、それぞれの初期構造に対して、アンブレラポテンシャルを課したMDシミュレーションを実行していきます。

## Step 1. ライブラリのインポートと環境設定
MDシミュレーションに必要なライブラリと、並列計算を行うための `joblib` を読み込みます。
また、PLUMEDをPythonから呼び出すための環境変数を設定します。

In [ ]:
import os
import sys
import numpy as np
from time import perf_counter
from joblib import Parallel, delayed

# ASE
from ase import units
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
# PLUMED wrapper
from ase.calculators.plumed import Plumed

# PFP (Matlantis)
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

# PLUMED Environment Variables (Please modify if necessary to match the path of your environment.)
plumed_path = "/home/jovyan/local/plumed-2.9.0"
os.environ["PLUMED_KERNEL"] = f"{plumed_path}/lib/libplumedKernel.so"
os.environ["PLUMED_TYPESAFE_IGNORE"] = "yes"
sys.path.append(plumed_path)

## Step 2. 計算パラメータの設定

シミュレーションの温度、時間、およびアンブレラサンプリングを行う反応座標の範囲を設定します。
また、`joblib` による並列数もここで指定します。

* **N_JOBS**: 同時に走らせる計算の数です。
* **colvars**: アンブレラサンプリングを行うターゲット距離のリストです（例: 13.0Å, 13.5Å, ...）。

In [ ]:
# PFP Settings
CALC_MODE     = "R2SCAN_PLUS_D3"
METHOD_TYPE   = "PFVM_D3_PFVM"
MODEL_VERSION = "v8.0.0"

# Joblib Settings
N_JOBS  = 10           # 並列数
VERBOSE = 10           # 進捗表示レベル
BACKEND = "threading"  # Matlantis利用時は "threading" 推奨

# MD Settings
TEMPERATURE  = 300.0    # Kelvin
TIMESTEP     = 1.0 * units.fs
TOTAL_STEPS  = 200_000
LOG_INTERVAL = 500

# Reaction Coordinate (Umbrella Windows)
colvars = np.arange(13.0, 32.01, 0.5)

## Step 3. 1つのウィンドウを計算する関数の定義

異なるアンブレラウィンドウのシミュレーションを`joblib` で並列化するために、1つのターゲット距離（ウィンドウ）に対してMDを実行する関数を定義します。

**PLUMEDの設定 (`plumed_setting`)のポイント:**
  * 単位系は`UNITS`でエネルギーeV、距離Åに指定しています。
  * `DISTANCE ATOMS=9,99`: 引っ張る原子のインデックスを指定します（※PLUMEDは1始まりなので、ASEのindex 8, 98 は 9, 99 になります）。
  * `RESTRAINT ... KAPPA=0.2 AT={cv_str}`: バネ定数 `0.2` で、ターゲット距離 `AT` に拘束します。

In [ ]:
def run_us_window(cv):
    """
    指定された反応座標 cv (距離) でアンブレラサンプリングを実行する関数
    """
    s_time = perf_counter()

    # --------------------------------------------------------
    # 1. Prepare Variables & Directories
    # --------------------------------------------------------
    cv_str = f"{cv:.2f}"
    out_dir = f"./output/04_umbrella_sampling/cv_{cv_str}"

    # フォルダ作成は計算開始前に必須
    os.makedirs(out_dir, exist_ok=True)

    # --------------------------------------------------------
    # 2. Prepare Calculator (Instantiate inside function)
    # --------------------------------------------------------
    estimator = Estimator(
        calc_mode=CALC_MODE,
        method_type=METHOD_TYPE,
        model_version=MODEL_VERSION
    )
    calculator = ASECalculator(estimator)

    # --------------------------------------------------------
    # 3. Prepare Atoms
    # --------------------------------------------------------
    input_xyz = f'./input/04_umbrella_sampling/initial_cv_{cv_str}.xyz'
    if not os.path.exists(input_xyz):
        input_xyz = f'./assets/04_umbrella_sampling/initial_cv_{cv_str}.xyz'

    atoms = read(input_xyz)

    # --------------------------------------------------------
    # 4. PLUMED Settings
    # --------------------------------------------------------
    # アンブレラポテンシャルの設定
    # 1. 原子間距離(dist)を定義 (indexは1-based)
    # 2. 調和ポテンシャル(RESTRAINT)で拘束: V(x) = 0.5 * KAPPA * (x - AT)^2
    plumed_setting = [f"UNITS LENGTH=A ENERGY=eV",
        # 1. Define distance
        "dist: DISTANCE ATOMS=9,99",

        # 2. Apply umbrella potential (https://www.plumed.org/doc-v2.9/user-doc/html/lugano-2.html)
        #    1eV = 96.48 kJ/mol
        f"restraint: RESTRAINT ARG=dist KAPPA=0.2 AT={cv_str}",

        # 3. Output
        f"PRINT STRIDE=500 ARG=dist,restraint.bias,restraint.force2 FILE={out_dir}/COLVAR_{cv_str}",

        # 1000ステップごとにバッファを強制書き込み
        "FLUSH STRIDE=1000"
    ]

    # Attach PLUMED calculator
    atoms.calc = Plumed(
        calc=calculator,
        input=plumed_setting,
        timestep=TIMESTEP,
        atoms=atoms,
        kT=units.kB * TEMPERATURE
    )

    # --------------------------------------------------------
    # 5. MD Simulation Setup
    # --------------------------------------------------------

    MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE, force_temp=True)
    Stationary(atoms)

    # Dynamics
    dyn = Langevin(
        atoms,
        TIMESTEP,
        temperature_K=TEMPERATURE,
        friction=0.002/units.fs,
        trajectory=f'{out_dir}/md-dyn.traj',
        logfile=f'{out_dir}/md-dyn.log',
        loginterval=LOG_INTERVAL
    )

    # --------------------------------------------------------
    # 6. Run Execution
    # --------------------------------------------------------
    dyn.run(TOTAL_STEPS)

    # Output Restart
    write(f'{out_dir}/md-dyn-restart.pdb', atoms)

    elapsed_time = perf_counter() - s_time
    return f"Done: cv={cv_str} ({elapsed_time:.1f} sec)"

## Step 4. アンブレラサンプリングの実行

`joblib.Parallel` を使用して、設定したすべてのウィンドウの計算を並列に実行します。

In [ ]:
# ============================================================
# 4. Main Execution
# ============================================================

print(f"Start Parallel Calculation: {len(colvars)} windows")
print(f"Settings: n_jobs={N_JOBS}, backend={BACKEND}")

# 並列実行
results = Parallel(n_jobs=N_JOBS, verbose=VERBOSE, backend=BACKEND)(
    delayed(run_us_window)(cv) for cv in colvars
)

In [ ]:
# 結果の表示
print("\n--- Results ---")
for res in results:
    print(res)

## 補足

本ノートブックでは、アンブレラサンプリングによるデータ収集を行いました。 正確な自由エネルギー計算を行うためには、反応座標上において隣接するウィンドウ間のヒストグラムが十分に重なりがあることが重要です。もしオーバーラップが不足している場合、解析時に誤差が生じたり、計算が収束しない原因となります。オーバラップが十分でない場合は、バネ定数 `KAPPA` を弱くするか、その間に新しいウィンドウを追加する必要があります。

## Next Step

次の[06_mbar_free_energy](./06_mbar_free_energy_ja.ipynb)では、いよいよ最後の仕上げとして自由エネルギー計算の手順を見ていきます。